# Homework 5: Coding

In the code spaces below, write code that answers each question. You can assume that the code spaces will all be executed in order, so any necessary libraries only need to be imported once. 

#### Exercise 1 
   
Refer to the `hw_05.pdf` file for a diagram of the quantum teleportation circuit. You are going to implement this circuit in Qiskit below. Write your code in the 'Your code goes here:' area in the cell below. The cell after that contains a loop that calls your function and tests it on a sequence of random qubit states. 

Note that this exercise makes use of the Statevector library for generating a random qubit state and then checking the received state to make sure that it is equal (up to a global phase) to the sent qubit. Remember that checking the state of an arbitrary quantum system is *only possible in simulation*. Actual qubits in unknown or arbitrary cannot simply be compared to each other, they can only be measured, which collapses the state vector to a simple 1 or 0. So this specific exercise could not be carried out as is on actual quantum hardware. 

In [2]:
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.quantum_info import Statevector, partial_trace, state_fidelity
from qiskit_aer import AerSimulator
# Use a seed to run repeatable simulations
# backend = AerSimulator(seed_simulator = 788026)
# Use no seed to have each simulation yield a randomized result
backend = AerSimulator()

sampler = Sampler(backend)

def teleport_qubit(alice_qb_state):
    # Set up the circuit
    qubits = QuantumRegister(3, name='q')
    cbits = ClassicalRegister(2, name='c')
    circuit = QuantumCircuit(qubits, cbits)

    # Use qubit 0 as Alice's sent qubit, 
    # qubit 1 as alices qubit in the shared entangled pair, 
    # and qubit 2 as Bob's qubit in the shared entangled pair 
    a_q, eq_a, eq_b = qubits
    x, z = cbits

    # Set the state of Alice's qubit in the circuit at 
    # qubit index 0
    circuit.initialize(alice_qb_state.data, a_q)
    
    ####################################################
    # TODO: Set up the necessary gates to teleport Alice's
    # qubit to Bob's
    # Your code goes here:
    circuit.h(eq_a)
    circuit.cx(eq_a,eq_b)
    circuit.cx(a_q, eq_a)
    circuit.h(a_q)
    circuit.cx(eq_a, eq_b)
    circuit.cz(a_q, eq_b)


    ######################################################
    # You do not need to make any changes below this point. 

    # Run the simulation
    circuit.save_statevector()
    simulator = AerSimulator(method='statevector')
    job = simulator.run(circuit)
    result = job.result()

    # We use some features here that we won't talk about until later in 
    # the course (density matrix, partial trace) but basically all the 
    # code is doing here is extracting the state of Bob's qubit from the
    # full state of the system at the end of the circuit. 
    # Extract the statevector
    full_state_vector = result.get_statevector()
    reduced_density_matrix = partial_trace(full_state_vector, [0,1])
    output_state = reduced_density_matrix.to_statevector()
    return output_state


/var/folders/8y/t4fp66zs4pn5dgw11026qw580000gn/T/ipykernel_40373/1562413663.py:10: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = Sampler(backend)


In [3]:
# You do not need to make any changes to this cell. Test your code above by executing
# this cell. The loop below runs the teleportation function 10 times. The 
# teleportation function must succeed all 10 times. 
from qiskit.quantum_info import random_statevector

transmitted_qubit_states = []
successful_transmissions = []

for _ in range(10):
    # Generate a random statevector as Alice's qubit to send
    alice_qb_state = random_statevector(2)

    # Carry out the teleportation 
    bobs_received_state = teleport_qubit(alice_qb_state)

    # Compare the sent qubit to the received qubit. It's not
    # straightfoward to eyeball this or use "==" because the states can
    # differ in global phase. We use .equiv() for comparing state vector 
    # objects. 
    is_equivalent = bobs_received_state.equiv(alice_qb_state) 

    transmitted_qubit_states.append(alice_qb_state)
    successful_transmissions.append(is_equivalent)
    
if all(successful_transmissions):
    print("****************************************************")
    print("* Successfully transmitted the sequence of states! *")
    print("****************************************************\n")
else:
    print("Failed to transmit all states in sequence:\n")

for state in transmitted_qubit_states:
        print(state)

****************************************************
* Successfully transmitted the sequence of states! *
****************************************************

Statevector([-0.05231511-0.68704788j,  0.23592062+0.68525163j],
            dims=(2,))
Statevector([0.93383257+0.16727737j, 0.27575357+0.15470933j],
            dims=(2,))
Statevector([ 0.38333976+0.28208481j, -0.78958722+0.38733811j],
            dims=(2,))
Statevector([-0.45315619-0.19231536j, -0.14462171-0.85834074j],
            dims=(2,))
Statevector([0.19889271-0.74585481j, 0.63269033+0.06200993j],
            dims=(2,))
Statevector([ 0.87742789+0.12217921j, -0.021155  +0.46340588j],
            dims=(2,))
Statevector([0.09725182+0.92335459j, 0.19201035-0.31794719j],
            dims=(2,))
Statevector([0.39116914+0.68677354j, 0.25270925-0.55809214j],
            dims=(2,))
Statevector([-0.1217909 +0.17171395j,  0.04227016-0.97667524j],
            dims=(2,))
Statevector([ 0.35903105-0.29122227j, -0.50995708-0.72541717j],
 

#### Exercise 2
   
Refer to the Week 3 demo code to see how to run workloads on actual quantum machinery. For this exercise, you will adapt the BB84 algorithm in this week's demo code to have it run on quantum machinery. 

You'll make a few modifications. First, you'll give Alice 150 bits to send:

```
alice_bits = [
     0, 1, 0, 1, 1, 0, 1, 0, 1, 1,
     0, 1, 0, 0, 1, 0, 1, 1, 0, 1,
     1, 0, 0, 0, 1, 1, 0, 0, 1, 0,
     0, 1, 1, 1, 1, 0, 1, 0, 1, 1,
     0, 1, 1, 0, 1, 0, 1, 1, 0, 0,
     0, 1, 1, 0, 1, 1, 0, 1, 0, 1,
     1, 1, 1, 1, 1, 0, 1, 0, 0, 1,
     0, 1, 1, 0, 1, 0, 1, 1, 0, 0,
     1, 0, 1, 0, 1, 1, 0, 1, 1, 0,
     0, 1, 0, 0, 1, 0, 1, 1, 0, 1,
     1, 0, 0, 0, 1, 1, 0, 0, 1, 0,
     0, 1, 1, 1, 1, 0, 1, 0, 1, 1,
     0, 1, 1, 0, 1, 0, 1, 1, 0, 0,
     0, 1, 1, 0, 1, 1, 0, 1, 0, 1,
     1, 1, 1, 1, 1, 0, 1, 0, 0, 1,
     0, 1, 1, 1, 1, 0, 1, 0, 1, 1,
     0, 1, 1, 0, 1, 0, 1, 1, 0, 0,
     0, 1, 1, 0, 1, 1, 0, 1, 0, 1,
     1, 1, 1, 1, 1, 0, 1, 0, 0, 1
    ]
```

Each of these qubits requires setting up a circuit, and we don't want to wait in the queue for quantum hardware 150 times. So, run a loop to create a a list of 150 transpiled circuits, and then run them all as a job together:

```
job = sampler.run(circuits, shots=1)
```

Note the difference from the Week 3 sample code: `circuits` is a list of transpiled circuits, and `shots` is set to 1. 

To access bob's qubits from this batch execution, loop through the result object and build up the `bob_bits` list in that way. 

```
result = job.result()
for i in range(len(alice_bits)):
    bitstring = result[i].data.meas.get_bitstrings()[0]
    bob_bits.append(int(bitstring))
```

Once you are able to run the 150-qubit job on physical hardware, manually re-run the job 10 times. For each time you re-run the experiment, make a note of how many bits were transmitted and how many, if any, mismatches there were between shared bits. Enter the averages over the 10 executions in the Markdown field below. 

Most of the time, you will not see any/many mismatches. However, with 10 executions of an 150-qubit sequence, I would expect you to see a small number of mismatches. If you have not seen any mismatches, double check that you are running on hardware and not a simulator with this code:

```
print(back.name, "  simulator:", back.simulator)
```

And continue manually running the experiment until you get at least one mismach between `alice` and `bob`. 

Note: each execution should take less than 30 seconds to complete. If your execution is taking longer than 30 seconds, interrupt and try again, making sure that you are submitting a single batch job. 



In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler.preset_passmanagers import(
    generate_preset_pass_manager)
from dotenv import load_dotenv
import random
import os
# For consistent results, uncomment this line
# random.seed(788026)

# You must set up an IBM Quantum account
# https://quantum.cloud.ibm.com/
# and create an API key. You also need to create an instance. Create an "Open" instance. Do not
# create a "Pay as you go" instance.

# The following lines load your API token from a .env file. This is NOT intended for version control,
# so .gitignore specifies to ignore .env files. I have included a file called "sample_dot_env"
# which shows you how the IBMQ_API_KEY can be set with an actual API key. Replace the fake
# API key in that file with your actual IBM Quantum API key and rename the file to .env (note the
# dot at the beginning). You should always keep API keys of any kind out of version control.

from getpass import getpass
token = getpass("IBM Quantum API Key: ")
service = QiskitRuntimeService(token=token, channel="ibm_quantum_platform")


# TODO:
# Your code starts here:
# Call your backend 'back' to work with the code below. 
back = service.least_busy(operational=True, simulator=False)
pm = generate_preset_pass_manager(backend=back, optimization_level=1)
sampler = SamplerV2(mode=back)

alice_bits = [
     0, 1, 0, 1, 1, 0, 1, 0, 1, 1,
     0, 1, 0, 0, 1, 0, 1, 1, 0, 1,
     1, 0, 0, 0, 1, 1, 0, 0, 1, 0,
     0, 1, 1, 1, 1, 0, 1, 0, 1, 1,
     0, 1, 1, 0, 1, 0, 1, 1, 0, 0,
     0, 1, 1, 0, 1, 1, 0, 1, 0, 1,
     1, 1, 1, 1, 1, 0, 1, 0, 0, 1,
     0, 1, 1, 0, 1, 0, 1, 1, 0, 0,
     1, 0, 1, 0, 1, 1, 0, 1, 1, 0,
     0, 1, 0, 0, 1, 0, 1, 1, 0, 1,
     1, 0, 0, 0, 1, 1, 0, 0, 1, 0,
     0, 1, 1, 1, 1, 0, 1, 0, 1, 1,
     0, 1, 1, 0, 1, 0, 1, 1, 0, 0,
     0, 1, 1, 0, 1, 1, 0, 1, 0, 1,
     1, 1, 1, 1, 1, 0, 1, 0, 0, 1,
     0, 1, 1, 1, 1, 0, 1, 0, 1, 1,
     0, 1, 1, 0, 1, 1, 0, 1, 0, 0,
     0, 1, 1, 0, 1, 1, 0, 1, 0, 1,
     1, 1, 1, 1, 1, 0, 1, 0, 0, 1
]
alice_bits = alice_bits[:150]
bob_bits = []
alice_used_h = []
bob_used_h = []
circuits = []
for bit in alice_bits:
    circuit = QuantumCircuit(1)
    if bit:
        circuit.x(0)
    alice_h=random.choice([True, False])
    if alice_h:
        circuit.h(0)
    circuit.barrier()
    bob_h = random.choice([True, False])
    if bob_h:
        circuit.h(0)
    circuit.measure_all()
    circuits.append(pm.run(circuit))
    alice_used_h.append(alice_h)
    bob_used_h.append(bob_h)
job=sampler.run(circuits, shots=1)
result=job.result()
for i in range(len(alice_bits)):
    bitstring = result[i].data.meas.get_bitstrings()[0]
    bob_bits.append(int(bitstring))
agree = [i for i in range(len(alice_bits)) if alice_used_h[i] == bob_used_h[i]]
alice = [alice_bits[i] for i in agree]
bob = [bob_bits[i] for i in agree]



# End of your code
#####################

if alice == bob:
    print("Key successfully shared!")
else:
    print("Key was not successfully shared")

mismatches = sum(a != b for a, b in zip(alice, bob))
qber = mismatches / len(alice) if alice else 0.0
print(f"Sifted bits: {len(alice)}, Mismatches: {mismatches}, QBER: {qber:.4f}")
print(back.name, "  simulator:", back.simulator)

qiskit_runtime_service._discover_account:WARNING:2026-11-11 10:51:54,280: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-11-11 10:51:56,651: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-11-11 10:51:57,162: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-11-11 10:51:59,004: Using instance: open-instance, plan: open
/var/folders/8y/t4fp66zs4pn5dgw11026qw580000gn/T/ipykernel_4

Key successfully shared!
Sifted bits: 87, Mismatches: 0, QBER: 0.0000
ibm_marrakesh   simulator: False


### Exercise 2 results:

Average number of transmitted bits over 10 executions: 78.6

Average number of mismatched bits over 10 executions: 1.7


Backend used most often: ibm_fez

Run 10 times and record the results before calculating averages.
